In [2]:
import plotly.graph_objects as go

def twh_to_ej(twh):
    """
    Convert Terawatt-hours (TWh) to Exajoules (EJ).

    Parameters:
    twh (float): Energy in Terawatt-hours.

    Returns:
    float: Energy in Exajoules.
    """
    conversion_factor = 0.0036
    return twh * conversion_factor

def xy(d):
    return zip(*sorted(d.items())) if d else ([], [])

* Data Source:
    * the 11th Basic Plan for Supply and Demand of Power (BPESD, `../resources/BPESD-11-20250313`)
    * 2023 Electricity Statistics of Korea (ES, `../resources/ES-93-KEPCO`)
    * [Renewable energy capacity projection discussed at the Electricity Policy Council meeting](https://www.epj.co.kr/news/articleView.html?idxno=31848)
    * Ministry of Trade, Industry and Energy (MOTIE), 2024, Roadmap for Competitive Bidding in Offshore Wind Power (RCBOWP, `../resources/RCBOWP-20240320-MOTIE`)

* Implemented Input Files
    * `/input/policy/korea-2035/power/onwnid_const_value.xml`
    * `/input/policy/korea-2035/power/onwind_const_techs.xml`
    * `/input/policy/korea-2035/power/offwind_const_value_cp.xml`
    * `/input/policy/korea-2035/power/offwind_const_value_ep.xml`
    * `/input/policy/korea-2035/power/offwind_const_techs.xml`

# Wind

The 11th Basic Plan presents capacity targets for total wind power but does not provide separate targets for onshore and offshore wind. Plans for offshore wind capacity were discussed at the Electricity Policy Council meeting in October 2022 during deliberations on the 10th Basic Plan. At that meeting, offshore wind capacity was projected at 0.1 GW in 2023, 1.3 GW in 2026, 14.3 GW in 2030, 21.8 GW in 2033, and 26.7 GW in 2036. In this study, it is assumed that these offshore wind capacity targets remain unchanged under the 11th Basic Plan. Onshore wind capacity is calculated by subtracting the offshore wind capacity from the total wind capacity targets.

In [3]:
dictCapGwOff = {2020: 0, 2023: 0.1, 2026: 1.3, 2030: 14.3, 2033: 21.8, 2036: 26.7}
dictCapGwOff[2025] = dictCapGwOff[2023] + (dictCapGwOff[2026] - dictCapGwOff[2023]) * (2/3)
dictCapGwOff[2035] = dictCapGwOff[2033] + (dictCapGwOff[2036] - dictCapGwOff[2033]) * (2/3)
dictCapGwOff

{2020: 0,
 2023: 0.1,
 2026: 1.3,
 2030: 14.3,
 2033: 21.8,
 2036: 26.7,
 2025: 0.8999999999999999,
 2035: 25.066666666666666}

In [4]:
dictCapGwOn = {2020: 1.64, 2023: 2.15, 2025: 3.02, 2030: 18.28, 2035: 32.97}
for year in dictCapGwOn.keys():
    dictCapGwOn[year] -= dictCapGwOff[year]
dictCapGwOn

{2020: 1.64,
 2023: 2.05,
 2025: 2.12,
 2030: 3.9800000000000004,
 2035: 7.903333333333332}

We use the Copernicus Weather Cutout data for South Korea (2013) to get capacity factors for onshore and offshore (onshore=0.372, offshore=0.218) for converting solar capacities into generation

In [5]:
dictCapTWhOff = {2020: 0.001, 2023: 0.35, 2025: 3.13, 2030: 49.66, 2035: 87.06}
dictCapTWhOn = {2020: 3.14, 2023: 3.38, 2025: 3.14 * 2.12/1.64, 2030: 3.38 * 3.98/1.64, 2035: 3.14 * 7.90 / 1.64}

According to the Roadmap for Competitive Bidding in Offshore Wind Power (RCBOWP) released by the Ministry of Trade, Industry and Energy (MOTIE), Korea plans to add 4 GW of offshore wind capacity annually during 2023–2025. In this study, we assume that this steady annual increase of 4 GW continues after 2025 through 2035.

In [8]:
dictCapGwOffEp = {2020: 0, 2023: 0.1, 2025: 0.90, 2030: 20.90, 2035: 40.90}
dictCapTWhOffEp = {2020: 0., 2023: 0.35, 2025: 3.13, 2030: 72.58, 2035: 142.03}

In [ ]:
# Sort years and values together (to avoid mismatching values)
years_cap, values_cap = zip(*sorted(dictCapOn.items()))  # Unpack sorted tuples
years_alt, values_alt = zip(*sorted(dictCapGwOff.items()))  # For the second dataset

# Create line plot
fig = go.Figure()

# Add first dataset (Coal Power Generation)
fig.add_trace(go.Scatter(x=years_cap, y=values_cap, mode='lines+markers', name='Onshore Wind', line_color='lightblue'))

# Add second dataset (Alternative Scenario)
fig.add_trace(go.Scatter(x=years_alt, y=values_alt, mode='lines+markers', name='Offshore Wind', line_color='blue'))

# Create annotations for specific years
annotations = []
for target_year in [2023, 2030, 2035]:
    # Only annotate if the year exists in the dictionary
    if target_year in dictCapGwOn:
        annotations.append(
            go.layout.Annotation(
                x=target_year,
                y=dictCapGwOn[target_year],
                xanchor='center',
                yanchor='bottom',
                text=f"{dictCapGwOn[target_year]:.1f} GW",
                showarrow=True,
                arrowhead=1,
                ax=0,
                ay=-20  # vertical offset for the text
            )
        )

    if target_year in dictCapGwOff:
        annotations.append(
            go.layout.Annotation(
                x=target_year,
                y=dictCapGwOff[target_year],
                xanchor='center',
                yanchor='bottom',
                text=f"{dictCapGwOff[target_year]:.1f} GW",
                showarrow=True,
                arrowhead=1,
                ax=0,
                ay=+40  # vertical offset for the text
            )
        )

# Update layout: set size, axis labels, template, and add annotations
fig.update_layout(
    xaxis_title='Year',
    xaxis_title_font_size=18,
    yaxis_title='TWh',
    yaxis_title_font_size=18,
    template='plotly_white',
    title_x=0.5,
    annotations=annotations,
    width=800,  # set figure width
    height=600   # set figure height
)
fig.update_xaxes(tickfont=dict(size=15))
fig.update_yaxes(tickfont=dict(size=15))

# Show figure interactively
fig.show()

# --- Optional: export a high-resolution image ---
# Note: requires 'kaleido' or 'orca' for static image export.
# Increase the 'scale' parameter for higher resolution (e.g., scale=2 or more).
fig.write_image("../../figure-power/wind.png", scale=2)

In the *Enhanced Ambition* scenario, solar capacities are tripled by 2030 compared to 2022 in line with [Korea’s commitment](https://www.mofa.go.kr/eng/wpge/m_5657/contents.do) to triple renewable energy capacity by 2030, announced during COP28. This annual rate of capacity increase is maintained through 2035.

In [8]:
63.6 + (63.6-31.964)

95.236

In [9]:
dictCapGwEp = {2020: 14.8, 2022: 21.2, 2025: 31.964, 2030: 63.6, 2035: 95.236}

In [10]:
dictCapTWhEp = {}
for year, gw in dictCapGwEp.items():
    dictCapTWhEp[year] = gw * 8.760 * 0.1406
dictCapTWhEp

{2020: 18.2285088,
 2022: 26.1111072,
 2025: 39.368652384,
 2030: 78.33332159999999,
 2035: 117.29799081600001}

In [ ]:
years_on, values_on = xy(dictCapTWhOn)
years_off, values_off = xy(dictCapTWhOff)
years_alt, values_alt = xy(dictCapTWhOffEp)

fig = go.Figure()

for name, x, y, dash in [
    ("Onshore", years_on, values_on, None),
    ("Offshore (Current)", years_off, values_off, None),
    ("Offshore (Enhanced)", years_alt, values_alt, "dash"),
]:
    fig.add_trace(go.Scatter(
        x=list(x), y=list(y),
        mode='lines+markers',
        name=name,
        line=(dict(dash=dash) if dash else None)
    ))

# Build annotations without repeating blocks
target_years = [2030, 2035]
annotations = []
for d in (dictCapTWhOn, dictCapTWhOff, dictCapTWhOffEp):
    for yr in target_years:
        val = d.get(yr)
        if val is not None:
            annotations.append(go.layout.Annotation(
                x=yr, y=val,
                xanchor='center', yanchor='bottom',
                text=f"{val:.1f} TWh",
                showarrow=True, arrowhead=1, ax=0, ay=-20
            ))

fig.update_layout(
    template='plotly_white',
    title_x=0.5,
    width=800, height=600,
    annotations=annotations,
    xaxis=dict(title='Year', title_font=dict(size=18), tickfont=dict(size=15)),
    yaxis=dict(title='TWh',  title_font=dict(size=18), tickfont=dict(size=15)),
)

fig.write_image("../figures/wind_generation.png", scale=2)
fig.show()

In [13]:
print(f"year = 2020, onshore wind (EJ) = {twh_to_ej(dictCapTWhOn[2020]):.4f}")
print(f"year = 2025, onshore wind (EJ) = {twh_to_ej(dictCapTWhOn[2025]):.4f}")
print(f"year = 2030, onshore wind (EJ) = {twh_to_ej(dictCapTWhOn[2030]):.4f}")
print(f"year = 2035, onshore wind (EJ) = {twh_to_ej(dictCapTWhOn[2035]):.4f}")

year = 2020, onshore wind (EJ) = 0.0113
year = 2025, onshore wind (EJ) = 0.0146
year = 2030, onshore wind (EJ) = 0.0295
year = 2035, onshore wind (EJ) = 0.0545


`/input/policy/korea-2035/power/onwind_const_value.xml`

```xml

<?xml version="1.0" ?>
<scenario>
  <world>
    <region name="South Korea">
      <policy-portfolio-standard name="Onwind-Generation-Ceiling">
        <policyType>tax</policyType>
        <market>South Korea</market>
        <min-price year="2020">0</min-price>
        <min-price year="2025">0</min-price>
        <min-price year="2030">0</min-price>
        <min-price year="2035">0</min-price>
        <constraint year="2020">0.0113</constraint>
        <constraint year="2025">0.0146</constraint>
        <constraint year="2030">0.0295</constraint>
        <constraint year="2035">0.0544</constraint>
      </policy-portfolio-standard>
    </region>
  </world>
</scenario>
```

In [15]:
print(f"year = 2020, offshore wind (EJ) = {twh_to_ej(dictCapTWhOff[2020]):.4f}")
print(f"year = 2025, offshore wind (EJ) = {twh_to_ej(dictCapTWhOff[2025]):.4f}")
print(f"year = 2030, offshore wind (EJ) = {twh_to_ej(dictCapTWhOff[2030]):.4f}")
print(f"year = 2035, offshore wind (EJ) = {twh_to_ej(dictCapTWhOff[2035]):.4f}")

year = 2020, offshore wind (EJ) = 0.0000
year = 2025, offshore wind (EJ) = 0.0113
year = 2030, offshore wind (EJ) = 0.1788
year = 2035, offshore wind (EJ) = 0.3134


`/input/policy/korea-2035/power/offwind_const_value_cp.xml`

```xml

<?xml version="1.0" ?>
<scenario>
  <world>
    <region name="South Korea">
      <policy-portfolio-standard name="Offwind-Generation-Floor">
        <policyType>subsidy</policyType>
        <market>South Korea</market>
        <min-price year="2020">0</min-price>
        <min-price year="2025">0</min-price>
        <min-price year="2030">0</min-price>
        <min-price year="2035">0</min-price>
        <constraint year="2020">0.0001</constraint>
        <constraint year="2025">0.0112</constraint>
        <constraint year="2030">0.1787</constraint>
        <constraint year="2035">0.3134</constraint>
      </policy-portfolio-standard>
    </region>
  </world>
</scenario>

```

In [16]:
print(f"year = 2020, offshore wind (EJ) = {twh_to_ej(dictCapTWhOffEp[2020]):.4f}")
print(f"year = 2025, offshore wind (EJ) = {twh_to_ej(dictCapTWhOffEp[2025]):.4f}")
print(f"year = 2030, offshore wind (EJ) = {twh_to_ej(dictCapTWhOffEp[2030]):.4f}")
print(f"year = 2035, offshore wind (EJ) = {twh_to_ej(dictCapTWhOffEp[2035]):.4f}")

year = 2020, offshore wind (EJ) = 0.0000
year = 2025, offshore wind (EJ) = 0.0113
year = 2030, offshore wind (EJ) = 0.2613
year = 2035, offshore wind (EJ) = 0.5113


`/input/policy/korea-2035/power/offwind_const_value_ep.xml`

```xml

<?xml version="1.0" ?>
<scenario>
  <world>
    <region name="South Korea">
      <policy-portfolio-standard name="Offwind-Generation-Floor">
        <policyType>subsidy</policyType>
        <market>South Korea</market>
        <min-price year="2020">0</min-price>
        <min-price year="2025">0</min-price>
        <min-price year="2030">0</min-price>
        <min-price year="2035">0</min-price>
        <constraint year="2020">0.0001</constraint>
        <constraint year="2025">0.0112</constraint>
        <constraint year="2030">0.2612</constraint>
        <constraint year="2035">0.5113</constraint>
      </policy-portfolio-standard>
    </region>
  </world>
</scenario>

```